In [ ]:
# For data manipulation
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization, Input
from tensorflow.keras.utils import image_dataset_from_directory
from keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# For data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ingore the warnings
import warnings
# importing the dataset from kaggle
import kagglehub

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
likhon148_animal_data_path = kagglehub.dataset_download('likhon148/animal-data')

print('Data source import complete.')


In [ ]:
print("TensorFlow version:", tf.__version__)
print("CUDA enabled:", tf.test.is_built_with_cuda())
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# For data manipulation
warnings.filterwarnings('ignore')

In [ ]:
# Load the image dataset from the directory using utils
# ds = image_dataset_from_directory('E:\DataScience Codanics\Kaggle\LAPTOP Items classification\PC Part Classification\Data')

dataset = keras.utils.image_dataset_from_directory(
    directory = likhon148_animal_data_path,
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256),
)

In [ ]:
shuffled = dataset #.shuffle(buffer_size=100).batch(10).repeat(2)

In [ ]:
dataset_size = len(shuffled)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

train_ds = shuffled.take(train_size)
val_ds = shuffled.skip(train_size).take(val_size)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# this function is sepreted label and images
def dataset_to_tensors(shuffled):
    images = []
    labels = []
    for img, lbl in shuffled.unbatch():
        images.append(img.numpy())
        labels.append(lbl.numpy())
    return tf.convert_to_tensor(images), tf.convert_to_tensor(labels)

In [ ]:
# x_train, y_train, x_test, y_test  dividing
x_train, y_train = dataset_to_tensors(train_ds) #dataset_to_tensors is function where we seprate  images and label
x_test, y_test = dataset_to_tensors(val_ds)

In [ ]:
import os
unique, count=np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 8))
plt.pie(count, labels=unique, autopct='%.0f%%')
plt.title('Target Class Distribution')
plt.plot()

In [ ]:
# Creating a function to visualize the images

def visualize_images(path, num_images=5):

    # Get a list of image filenames
    image_filenames = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]

    #if not image_filenames:
        #raise ValueError("No images found in the specified path")

    # Select random images
    selected_images = random.sample(image_filenames, min(num_images, len(image_filenames)))

    # Create a figure and axes
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3), facecolor='white')

    # Display each image
    for i, image_filename in enumerate(selected_images):
        # Load image
        image_path = os.path.join(path, image_filename)
        image = plt.imread(image_path)

        # Display image
        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(image_filename)  # Set image filename as title

    # Adjust layout and display
    plt.tight_layout()
    plt.show()

In [ ]:
# Extrating the class labels
classes = shuffled.class_names

print(f'{classes}')

In [ ]:
import os
# Iterating through each class to plot its images
for label in classes:
    # Specify the path containing the images to visualize
    path_to_visualize = os.path.join(likhon148_animal_data_path,label)

    # Visualize 5 random images
    print(label.upper())
    visualize_images(path_to_visualize, num_images=5)

## **Min-Max Scalling**

In [ ]:
# Convert the data type of the images to float32
x_train = tf.cast(x_train, tf.float32)  # tf.cast is used to convert the image tensors to float32.
x_test = tf.cast(x_test, tf.float32)

# Normalize the pixel values to a range between 0 and 1
x_train /= 255.0
x_test /= 255.0  #The pixel values of the images are normalized by dividing by 255.0.

# Print the shapes of the original training data and labels
print("X_train shape: {} \nY_train shape: {}".format(x_train.shape, y_train.shape))
print("X_test shape: {} \nY_test shape: {}".format(x_test.shape, y_test.shape))

# **Processing the Target variable**

In [ ]:
# Convert class vectors to binary class matrices / one-hot encoding
y_train = to_categorical(y_train, 15)
y_test = to_categorical(y_test, 15)

In [ ]:
y_train[3]

### Define CNN Architecture

In [ ]:
# Define the CNN model
model = Sequential()
# Conv2D layer with 32 filters, kernel size 3x3, input shape (32, 32, 3)
# Input size: 32x32x3, Kernel size: 3x3, Number of kernels: 32, Output size: 30x30x32. [input_size-kernel]+1
model.add(Input(shape=(256, 256, 3)))
#model.add(Conv2D(32, 5, strides=2, activation="relu"))
#model.add(Conv2D(32, 3, activation="relu"))
#model.add(MaxPooling2D(3))
#model.add(BatchNormalization())
#model.add(Dense(128, activation='relu'))
#model.add(BatchNormalization())
#model.add(Dropout(0.5))
#model.add(Flatten())
#model.add(Dense(15, activation='softmax'))

model.add(Conv2D(32, 5, activation="relu"))
model.add(Conv2D(32, 3, activation="relu"))

model.add(Conv2D(64, 5, activation="relu"))
model.add(Conv2D(64, 3, activation="relu"))

model.add(Conv2D(54, (3, 3), activation='relu'))
model.add(BatchNormalization())
# MaxPooling2D layer with pool size 2x2
# Output size: 6x6x64
model.add(MaxPooling2D((2, 2)))

model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(MaxPooling2D(3, 3))
model.add(Flatten())
model.add(Dense(32, activation='relu'))
model.add(Dense(15, activation='softmax'))

model.summary()

# MaxPooling2D layer with pool size 2x2
# Output size: 15x15x32
# model.add(Dropout(0.25))

#model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

# Conv2D layer with 64 filters, kernel size 3x3
# Input size: 15x15x32, Kernel size: 3x3, Number of kernels: 64, Output size: 13x13x64
#model.add(Conv2D(84, (3, 3), activation='relu'))#
#model.add(BatchNormalization())
# MaxPooling2D layer with pool size 2x2
# Output size: 6x6x64
#model.add(MaxPooling2D((2, 2)))
#model.add(Dropout(0.25))

# Conv2D layer with 64 filters, kernel size 3x3
# Input size: 15x15x32, Kernel size: 3x3, Number of kernels: 64, Output size: 13x13x64
#model.add(Conv2D(64, (3, 3), activation='relu'))#
#model.add(BatchNormalization())
# MaxPooling2D layer with pool size 2x2
# Output size: 6x6x64
#model.add(MaxPooling2D((2, 2)))
#model.add(Dropout(0.25))




# Conv2D layer with 64 filters, kernel size 3x3
# Input size: 15x15x32, Kernel size: 3x3, Number of kernels: 64, Output size: 13x13x64
#model.add(Conv2D(54, (3, 3), activation='relu'))
#model.add(BatchNormalization())
# MaxPooling2D layer with pool size 2x2
# Output size: 6x6x64
#model.add(MaxPooling2D((2, 2)))
#model.add(Dropout(0.25))

# Conv2D layer with 64 filters, kernel size 3x3
# Input size: 15x15x32, Kernel size: 3x3, Number of kernels: 64, Output size: 13x13x64
#model.add(Conv2D(54, (3, 3), activation='relu'))
#model.add(BatchNormalization())
# MaxPooling2D layer with pool size 2x2
# Output size: 6x6x64
#model.add(MaxPooling2D((2, 2)))
#model.add(Dropout(0.25))

# Conv2D layer with 64 filters, kernel size 3x3
# Input size: 15x15x32, Kernel size: 3x3, Number of kernels: 64, Output size: 13x13x64#
#model.add(Conv2D(64, (3, 3), activation='relu'))
#model.add(BatchNormalization())
# MaxPooling2D layer with pool size 2x2
# Output size: 6x6x64
#model.add(MaxPooling2D((2, 2)))#
#model.add(Dropout(0.25))
# Conv2D layer with 64 filters, kernel size 3x3
# Input size: 6x6x64, Kernel size: 3x3, Number of kernels: 64, Output size: 4x4x64


# Flatten layer
# Output size: 1024

#model.add(Flatten())
# Dense layer with 64 units
# Input size: 1024, Output size: 64
#model.add(Dense(128, activation='relu'))
#model.add(BatchNormalization())
#model.add(Dropout(0.5))

# -----temporary disabled----
#model.add(Dense(128, activation='relu'))
#model.add(BatchNormalization())
#model.add(Dropout(0.5))
#------ and replaced by:

# Dense layer with 15 units (output layer)
# Input size: 256, Output size: 15
#model.add(Flatten())
#model.add(Dense(15, activation='softmax'))
#model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam


In [ ]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#other optimizer SGD, RMSprop, Adadelta, Adagrad, Adamax etc
# other loss functions are mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, mean_squared_logarithmic_error, binary_crossentropy

In [ ]:
# Train the model
#history = model.fit(x_train, y_train, batch_size=32, epochs=120,
          #validation_data=(x_test, y_test))
history = model.fit(x_train, y_train, batch_size=32, epochs=80,
          validation_data=(x_test, y_test)) # look if this is really needed test data are for prediction

In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(x_test, y_test)
print('Test accuracy:', accuracy)

In [ ]:
pred = model.predict(x_test)


In [ ]:

num_images_to_display = 40
num_columns = 4
num_rows = (num_images_to_display + num_columns - 1) // num_columns

fig, axes = plt.subplots(num_rows, num_columns, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    if i < num_images_to_display:
        ax.imshow(x_test[i])
        actual_label = classes[np.argmax(y_test[i])]
        print(f"A:{actual_label}")
        predicted_label = classes[np.argmax(pred[i])]
        print(f"P:{predicted_label}")
        ax.set_title(f"A:{actual_label}//P:{predicted_label}")
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

# **Plotting the Graph**


In [ ]:
# Plotting the graph of Accuracy and Validation Accuracy
plt.title('Training Accuracy vs Validation Accuracy')

plt.plot(history.history['accuracy'], color='red',label='Train')
plt.plot(history.history['val_accuracy'], color='blue',label='Validation')

plt.legend()

In [ ]:
# Plotting the graph of Accuracy and Validation loss
plt.title('Training Loss vs Validation Loss')

plt.plot(history.history['loss'], color='red',label='Train')
plt.plot(history.history['val_loss'], color='blue',label='Validation')

plt.legend()